# EXPERIMENT 9: BINARY ASK & BFSK MODULATION AND DEMODULATION
### Software-Based Digital Communication Laboratory (EC592)
**Institution:** Cooch Behar Government Engineering College, Department of Electronics & Communication Engineering  
**Presenter / Student Name:** Apurba Maity | **University Roll No.:** 34900324001 | **Semester:** 5th Sem ECE  

---

## 1. Laboratory Objectives
1. **Bandpass Digital Transmission:** Model the transmission of binary data over passband physical channels using sinusoidal carrier modulation.
2. **Binary Amplitude Shift Keying (BASK / OOK):** Mathematically formulate On-Off Keying, construct the 1-dimensional orthonormal basis $\phi_1(t)$, derive optimum decision threshold $\gamma = \sqrt{E_b/2}$, and evaluate theoretical error probability $Q\left(\sqrt{E_b / (2 N_0)}\right)$.
3. **Binary Frequency Shift Keying (BFSK):** Formulate dual-tone signaling across orthogonal carriers $f_1$ (Space) and $f_2$ (Mark), establish the 2-dimensional signal space, and prove the Euclidean distance $d = \sqrt{2 E_b}$.
4. **Orthogonality Conditions:** Analytically prove the cross-correlation coefficient $\rho = \mathrm{sinc}(2 \Delta f T_b)$, identifying the minimum coherent spacing $\Delta f = \frac{1}{2 T_b}$ (MSK) and non-coherent spacing $\Delta f = \frac{1}{T_b}$ (Sunde's BFSK).
5. **Receiver Architectures:** Contrast coherent correlation/matched-filter receivers against non-coherent envelope detection receivers.
6. **Envelope Statistics:** Derive and visualize Rayleigh distribution for noise-only intervals and Rician distribution for carrier-plus-noise intervals.
7. **Power Spectral Density (PSD):** Evaluate transmission bandwidth via Welch spectral estimation and verify Carson's rule for BFSK.
8. **Monte Carlo Bit Error Rate (BER):** Transmit $200,000$ bits across $E_b/N_0 \in [0, 12]\,\mathrm{dB}$, verifying theoretical bounds and quantifying the $3.0\,\mathrm{dB}$ power penalty of BFSK vs BPSK, and $6.0\,\mathrm{dB}$ penalty of BASK.

---

## 2. Core Python Setup & Global Parameters
We import standard scientific computation libraries: NumPy, SciPy, and Matplotlib.

In [ ]:
import os
import numpy as np
from scipy import signal, special
import matplotlib.pyplot as plt

# Global Parameters
RB = 1000.0          # Bit rate: 1000 bps
TB = 1.0 / RB        # Bit duration: 1.0 ms
FS = 100000.0        # Sampling frequency: 100 kHz
DT = 1.0 / FS
SAMPLES_PER_BIT = int(FS * TB)

FC_BASK = 4000.0     # BASK carrier: 4 kHz
F1_BFSK = 3000.0     # BFSK Space '0': 3 kHz
F2_BFSK = 5000.0     # BFSK Mark '1': 5 kHz

print(f'System Configuration: Rb = {RB} bps, Tb = {TB*1000} ms, Fs = {FS/1000} kHz')
print(f'BASK Carrier = {FC_BASK/1000} kHz')
print(f'BFSK Space = {F1_BFSK/1000} kHz, Mark = {F2_BFSK/1000} kHz, Delta f = {(F2_BFSK - F1_BFSK)/1000} kHz')

## 3. Time-Domain Signaling Waveforms
We synthesize the time-domain waveforms for an 8-bit sequence `[1, 0, 1, 1, 0, 0, 1, 0]` across BASK and Continuous-Phase BFSK.

In [ ]:
bits = np.array([1, 0, 1, 1, 0, 0, 1, 0])
n_bits = len(bits)
t = np.linspace(0, n_bits * TB, n_bits * SAMPLES_PER_BIT, endpoint=False)
t_ms = t * 1000.0

baseband = np.repeat(bits, SAMPLES_PER_BIT)
Ac = np.sqrt(2.0 / TB)

# BASK
bask_sig = baseband * Ac * np.cos(2 * np.pi * FC_BASK * t)

# BFSK (Continuous Phase)
bfsk_sig = np.zeros_like(t)
for i, b in enumerate(bits):
    idx = slice(i * SAMPLES_PER_BIT, (i + 1) * SAMPLES_PER_BIT)
    freq = F2_BFSK if b == 1 else F1_BFSK
    bfsk_sig[idx] = Ac * np.cos(2 * np.pi * freq * t[idx])

plt.figure(figsize=(12, 6))
plt.subplot(3, 1, 1)
plt.step(t_ms, baseband, where='post', color='#0f2b48', lw=2)
plt.title('(a) Binary Bitstream m(t)')
plt.ylabel('m(t)')
plt.grid(True)

plt.subplot(3, 1, 2)
plt.plot(t_ms, bask_sig, color='#b91c1c', lw=1.5)
plt.title('(b) BASK / OOK Modulated Signal')
plt.ylabel('s_BASK(t)')
plt.grid(True)

plt.subplot(3, 1, 3)
plt.plot(t_ms, bfsk_sig, color='#15803d', lw=1.5)
plt.title('(c) BFSK Modulated Signal')
plt.ylabel('s_BFSK(t)')
plt.xlabel('Time [ms]')
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. Signal Space Representations & Optimum Decision Boundaries
BASK is 1-dimensional ($N = 1$); BFSK is 2-dimensional ($N = 2$) with Euclidean distance $d = \sqrt{2 E_b}$.

In [ ]:
# Visualizing 2D BFSK Signal Space
Eb = 1.0
s0 = np.array([np.sqrt(Eb), 0.0])
s1 = np.array([0.0, np.sqrt(Eb)])

plt.figure(figsize=(6, 6))
plt.scatter([s0[0]], [s0[1]], color='#b45309', s=150, label='s0 = [sqrt(Eb), 0] (Space)')
plt.scatter([s1[0]], [s1[1]], color='#15803d', s=150, label='s1 = [0, sqrt(Eb)] (Mark)')
plt.plot([0, 1.5], [0, 1.5], 'r--', lw=2, label='Decision Boundary: r2 = r1')
plt.plot([s0[0], s1[0]], [s0[1], s1[1]], 'm:', lw=2, label='Euclidean Distance d = sqrt(2 Eb)')
plt.xlabel('phi_1(t) (Space Tone)')
plt.ylabel('phi_2(t) (Mark Tone)')
plt.title('BFSK 2D Orthogonal Signal Space')
plt.xlim(-0.2, 1.5)
plt.ylim(-0.2, 1.5)
plt.grid(True)
plt.legend()
plt.show()

## 5. BFSK Orthogonality Analysis
The cross-correlation $\rho = \mathrm{sinc}(2 \Delta f T_b)$ vanishes at $\Delta f = \frac{1}{2 T_b}$ (coherent MSK) and $\Delta f = \frac{1}{T_b}$ (non-coherent Sunde's FSK).

In [ ]:
delta_f_norm = np.linspace(0.01, 3.0, 500)
rho_coh = np.sinc(2 * delta_f_norm)
rho_noncoh = np.abs(np.sinc(delta_f_norm))

plt.figure(figsize=(10, 5))
plt.plot(delta_f_norm, rho_coh, 'b-', lw=2.2, label='Coherent Cross-Correlation')
plt.plot(delta_f_norm, rho_noncoh, 'r--', lw=2.0, label='Non-Coherent Worst-Case Envelope')
plt.axhline(0, color='k', lw=1)
plt.axvline(0.5, color='g', ls=':', label='MSK Coherent Spacing (0.5 Rb)')
plt.axvline(1.0, color='c', ls=':', label='Sunde Non-Coherent Spacing (1.0 Rb)')
plt.xlabel('Normalized Frequency Separation Δf * Tb')
plt.ylabel('Cross-Correlation Coefficient ρ')
plt.title('Orthogonality vs Frequency Separation')
plt.grid(True)
plt.legend()
plt.show()

## 6. Monte Carlo Bit Error Rate Simulation
We execute a Monte Carlo BER simulation transmitting $100,000$ bits per SNR point, comparing Coherent BASK, Coherent BFSK, Non-Coherent BFSK, and BPSK.

In [ ]:
snr_db = np.arange(0, 11, 1)
snr_lin = 10.0 ** (snr_db / 10.0)

# Theoretical Curves
ber_bpsk_th = 0.5 * special.erfc(np.sqrt(snr_lin))
ber_bfsk_coh_th = 0.5 * special.erfc(np.sqrt(snr_lin / 2.0))
ber_bfsk_noncoh_th = 0.5 * np.exp(-0.5 * snr_lin)
ber_bask_coh_th = 0.5 * special.erfc(np.sqrt(snr_lin / 4.0))

# Monte Carlo Simulation (100,000 bits)
n_sim = 100000
tx = np.random.randint(0, 2, n_sim)
sim_bfsk_coh = []
sim_bfsk_noncoh = []

for snr in snr_lin:
    sigma = np.sqrt(1.0 / (2.0 * snr))
    # Coherent BFSK
    r_space = (tx == 0).astype(float) + np.random.normal(0, sigma, n_sim)
    r_mark = (tx == 1).astype(float) + np.random.normal(0, sigma, n_sim)
    sim_bfsk_coh.append(np.mean(tx != (r_mark > r_space)))
    
    # Non-Coherent BFSK
    env_space = np.sqrt(((tx == 0).astype(float) + np.random.normal(0, sigma, n_sim))**2 + np.random.normal(0, sigma, n_sim)**2)
    env_mark = np.sqrt(((tx == 1).astype(float) + np.random.normal(0, sigma, n_sim))**2 + np.random.normal(0, sigma, n_sim)**2)
    sim_bfsk_noncoh.append(np.mean(tx != (env_mark > env_space)))

plt.figure(figsize=(10, 7))
plt.semilogy(snr_db, ber_bpsk_th, 'k:', lw=1.8, label='Theoretical BPSK: Q(sqrt(2 Eb/N0))')
plt.semilogy(snr_db, ber_bfsk_coh_th, 'b-', lw=2.2, label='Theoretical Coherent BFSK: Q(sqrt(Eb/N0))')
plt.semilogy(snr_db, ber_bfsk_noncoh_th, 'g--', lw=2.0, label='Theoretical Non-Coherent BFSK: 0.5 exp(-Eb/(2 N0))')
plt.semilogy(snr_db, ber_bask_coh_th, 'r-', lw=2.0, label='Theoretical Coherent BASK: Q(sqrt(Eb/(2 N0)))')
plt.semilogy(snr_db, sim_bfsk_coh, 'bo', label='Monte Carlo Coherent BFSK')
plt.semilogy(snr_db, sim_bfsk_noncoh, 'g^', label='Monte Carlo Non-Coherent BFSK')
plt.xlabel('Eb / N0 [dB]')
plt.ylabel('Bit Error Rate (BER)')
plt.title('Monte Carlo BER Performance Benchmark')
plt.grid(True, which='both')
plt.ylim(1e-5, 0.5)
plt.legend()
plt.show()

## 7. Conclusions & Summary
1. **BASK** offers simplicity but suffers poor power efficiency ($6\,\mathrm{dB}$ penalty vs BPSK) due to carrier power waste and extreme vulnerability to channel amplitude fading.
2. **BFSK** maintains a constant envelope, providing immunity to amplitude fading and compatibility with efficient non-linear power amplifiers.
3. **Orthogonality** guarantees zero inter-tone cross-talk at $\Delta f = R_b / 2$ for coherent detection and $\Delta f = R_b$ for non-coherent detection.
4. **Non-Coherent BFSK** exhibits less than $1\,\mathrm{dB}$ SNR penalty compared to coherent BFSK at practical BERs ($10^{-4}$), making it the optimal choice for battery-powered IoT and wireless sensors.